<a href="https://colab.research.google.com/github/RVC-Boss/GPT-SoVITS/blob/main/Colab-WebUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-SoVITS WebUI

## Env Setup (Run Once Only)
## 环境配置, 只需运行一次

### 1.

In [ ]:
%%writefile /content/setup.sh
set -e

cd /content

RESUME_INSTALL=false

if [ -d GPT-SoVITS/.git ]; then
    echo "GPT-SoVITS already exists, reusing existing checkout"
    RESUME_INSTALL=true
elif [ -e GPT-SoVITS ]; then
    echo "GPT-SoVITS exists but is not a git checkout; remove /content/GPT-SoVITS or choose another path" >&2
    exit 1
else
    git clone https://github.com/unknow857/GPT-SoVITS.git
fi

cd GPT-SoVITS

create_env() {
    conda create -n GPTSoVITS -c conda-forge python=3.10 "python_abi=3.10=*_cp310" pip -y
}

is_cpython_env() {
    conda run -n GPTSoVITS python -c 'import sys; raise SystemExit(0 if getattr(sys.implementation, "name", "").lower() == "cpython" else 1)'
}

if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    if is_cpython_env; then
        echo "GPTSoVITS uses CPython, reusing environment"
    else
        echo "GPTSoVITS is not CPython, recreating environment"
        conda env remove -n GPTSoVITS -y
        create_env
    fi
else
    create_env
fi

source activate GPTSoVITS

python -m pip install ipykernel
python -m ipykernel install --user --name GPTSoVITS --display-name "Python (GPTSoVITS)"

if [ "$RESUME_INSTALL" = true ]; then
    WORKFLOW=true bash install.sh --device CU126 --source HF --download-uvr5
else
    bash install.sh --device CU126 --source HF --download-uvr5
fi

python -m pip show torch
python -c 'import torch; print(f"torch {torch.__version__}, cuda={torch.version.cuda}")'
python -c 'import psutil; print(f"psutil {psutil.__version__}")'

### 2.

In [ ]:
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")
!cd /content && bash setup.sh

## Launch WebUI
## 启动 WebUI

In [ ]:
!cd /content/GPT-SoVITS && (conda run -n GPTSoVITS python -c "import psutil" || conda run --no-capture-output -n GPTSoVITS python -m pip install --force-reinstall --no-cache-dir psutil) && export is_share=True && conda run --no-capture-output -n GPTSoVITS python webui.py